# 04 - PhoBERT Extractive Sanity Notebook

Notebook goals:
- Validate `phobert-extractive` behavior on a small subset before the official benchmark.
- Compare quickly against `tfidf` and `textrank` under the same preprocessing pipeline.
- Inspect qualitative outputs and core metrics (ROUGE/compression/repetition).

In [1]:
from __future__ import annotations
import os 
import sys
import time
from pathlib import Path

prefix = Path(sys.prefix)
for dll_dir in [
    prefix,
    prefix / "Library" / "bin",
    prefix / "Lib" / "site-packages" / "pyarrow",
    prefix / "Lib" / "site-packages" / "pyarrow.libs",
]:
    if dll_dir.exists():
        os.add_dll_directory(str(dll_dir))

import pandas as pd

ROOT = Path.cwd().resolve().parents[0]
if (ROOT / "backend").exists() is False:
    ROOT = Path.cwd().resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "backend") not in sys.path:
    sys.path.insert(0, str(ROOT / "backend"))
if str(ROOT / "evaluation") not in sys.path:
    sys.path.insert(0, str(ROOT / "evaluation"))

from app.services.input import process_from_text
from app.services.summarization.summary_service import summarize_processed_input_raw
from evaluation.evaluator import Evaluator
from scripts.shared.io_dataset import load_benchmark_validation_subset, load_split

In [2]:
PROTOCOL_VERSION_EXPECTED = "phase0_v2"
TARGET_SPLIT = "validation"
SEED = 42
SUBSET_LIMIT = 30
ARTICLE_CHAR_THRESHOLD = 1200
TOP_K = 3
ENGINES = ["tfidf", "textrank", "phobert-extractive"]

processed_dir = ROOT / "data" / "processed" / "vietnews"
df, manifest = load_split(processed_dir, TARGET_SPLIT, PROTOCOL_VERSION_EXPECTED)
subset_df, protocol_version, manifest_path, split_path = load_benchmark_validation_subset(
    df,
    manifest=manifest,
    processed_dir=processed_dir,
    target_split=TARGET_SPLIT,
    seed=SEED,
    subset_limit=SUBSET_LIMIT,
    article_char_threshold=ARTICLE_CHAR_THRESHOLD,
)
subset_df[["guid", "article_char_len", "reference_char_len"]].head()

,guid,article_char_len,reference_char_len
0,494,1598,142
1,8563,2618,114
2,55,1977,148
3,11152,1964,115
4,5005,4410,169


In [3]:
evaluator = Evaluator(use_stemmer=False)
records = []

for engine in ENGINES:
    for row in subset_df.to_dict(orient="records"):
        article = str(row.get("article", ""))
        reference = str(row.get("reference_summary", ""))
        if not article.strip() or not reference.strip():
            continue

        processed = process_from_text(article)
        t0 = time.perf_counter()
        selected_sentences, engine_meta = summarize_processed_input_raw(
            processed,
            max_sentences=TOP_K,
            ratio=None,
            engine_name=engine,
        )
        latency_sec = time.perf_counter() - t0
        predicted_summary = " ".join(s.strip() for s in selected_sentences if s.strip())
        bundle = evaluator.evaluate_one(
            source_text=processed.cleaned_text,
            reference_summary=reference,
            predicted_summary=predicted_summary,
            latency_sec=latency_sec,
            extra={
                "guid": row.get("guid"),
                "engine": engine,
                "predicted_summary": predicted_summary,
                "selected_indices": engine_meta.get("selected_indices", []),
            },
        )
        records.append(bundle.as_dict())

detail_df = pd.DataFrame(records)
summary_df = (
    detail_df.groupby("engine", as_index=False)
    .agg(
        n=("engine", "count"),
        rouge1_f=("rouge1_f", "mean"),
        rouge2_f=("rouge2_f", "mean"),
        rougeL_f=("rougeL_f", "mean"),
        latency_sec=("latency_sec", "mean"),
        compression_ratio=("compression_ratio", "mean"),
        repetition_rate=("repetition_rate", "mean"),
    )
    .sort_values("rougeL_f", ascending=False)
    .reset_index(drop=True)
)
summary_df

c:\Users\OS\miniconda3\envs\vietsum\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 19494.58it/s]
RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those 

,engine,n,rouge1_f,rouge2_f,rougeL_f,latency_sec,compression_ratio,repetition_rate
0,textrank,30,0.425287,0.211777,0.284344,0.003120,0.204086,0.108329
1,phobert-extractive,30,0.394532,0.207605,0.264641,0.704708,0.232742,0.062917
2,tfidf,30,0.449440,0.156457,0.258428,0.000478,0.166488,0.013436


In [4]:
sample_guid = detail_df[detail_df["engine"] == "phobert-extractive"].head(1)["guid"].tolist()
if sample_guid:
    g = sample_guid[0]
    print("GUID:", g)
    source_row = subset_df[subset_df["guid"].astype(str) == str(g)].head(1).to_dict(orient="records")[0]
    print("\nArticle snippet:")
    print(str(source_row.get("article", ""))[:800])
    print("\nReference:")
    print(str(source_row.get("reference_summary", ""))[:500])
    print("\nPredictions:")
    for engine in ENGINES:
        pred = detail_df[(detail_df["guid"].astype(str) == str(g)) & (detail_df["engine"] == engine)].head(1)
        if not pred.empty:
            print(f"- {engine}:", str(pred.iloc[0]["predicted_summary"])[:500])

GUID: 494

Article snippet:
Tối 13/5 , thông_tin từ phòng Cảnh_sát hình_sự , Công_an tỉnh Đắk_Lắk cho biết , chiều cùng ngày đơn_vị đã đưa đối_tượng Nguyễn Hoàng Quân_Nhi , 23 tuổi , nhân_viên công_ty đòi nợ thuê ở TP. HCM gặp mặt , xin lỗi anh Nguyễn Sỹ Đức ( PV truyền hình ANTV tại Đắk Lắk ) . Trước đó , chiều 4/5 anh Đức liên tục nhận được điện thoại từ 4 số lạ gọi đến với nội dung đe dọa đến bản thân anh và gia đình của anh . Theo anh Đức , thời_gian qua , đã đưa tin , phản ánh nhiều vụ việc phức tạp , tiêu_cực , ảnh hưởng đến quyền lợi của một số cá nhân , tổ_chức nên nghi_ngờ bị kẻ xấu đe_đoạ , trả_thù . Do_đó , anh Đức đã đến phòng Cảnh sát hình sự Công_an tỉnh Đắk_Lắk trình_báo . Sau khi tiếp_nhận trình_báo , bằng các biện_pháp nghiệp lực_lượng Công_an tỉnh Đắk_Lắk xác_định được đối tượng đe_doạ anh Đức là Ng

Reference:
Một đối_tượng là nhân_viên đòi nợ thuê khai xác_minh nhầm phóng_viên ANTV là con_nợ nên đã nhiều lần gọi điện đe_doạ gia_đình phóng_viên này .

Predictions:
- 